# Predicting Income Level from Census Data
**Project 3 — Machine Learning Foundations**

## 1. Problem Definition

**Task type:** Supervised learning — **binary classification**.

**Goal:** Predict whether a person's annual income is **above \$50K** (`>50K`) or **at or below \$50K** (`<=50K`) from demographic and employment attributes recorded in the 1994 U.S. Census.

**Dataset:** *Adult* (also known as "Census Income"), UCI Machine Learning Repository — https://archive.ics.uci.edu/dataset/2/adult (Becker & Kohavi, 1996).
The original `adult.data` (train) and `adult.test` files were combined into a single `adult.csv` with a header row; no values were altered.

**Why this framing:** the target is a clearly defined categorical label, the classes are imbalanced (which makes metric selection meaningful), and the data contains sensitive attributes (sex, race) that allow us to examine model fairness.

## 2. Setup
Import libraries and fix a random seed so results are reproducible.

In [1]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

RANDOM_STATE = 42  # used everywhere randomness is involved, for reproducibility

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

print(f"numpy {np.__version__} | pandas {pd.__version__} | scikit-learn {sklearn.__version__}")

numpy 2.5.5 | pandas 3.1.1 | scikit-learn 1.9.1


## 3. Load and Inspect the Dataset

In [2]:
# Load the raw CSV. skipinitialspace guards against stray spaces after commas.
df = pd.read_csv("adult.csv", skipinitialspace=True)

print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Shape: 48,842 rows x 15 columns


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


### 3.1 Column names and data types

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             48842 non-null  int64
 1   workclass       48842 non-null  str  
 2   fnlwgt          48842 non-null  int64
 3   education       48842 non-null  str  
 4   education-num   48842 non-null  int64
 5   marital-status  48842 non-null  str  
 6   occupation      48842 non-null  str  
 7   relationship    48842 non-null  str  
 8   race            48842 non-null  str  
 9   sex             48842 non-null  str  
 10  capital-gain    48842 non-null  int64
 11  capital-loss    48842 non-null  int64
 12  hours-per-week  48842 non-null  int64
 13  native-country  48842 non-null  str  
 14  income          48842 non-null  str  
dtypes: int64(6), str(9)
memory usage: 5.6 MB


In [4]:
# Split columns by type — this drives preprocessing later
TARGET = "income"
numeric_cols = df.select_dtypes(include="number").columns.tolist()
categorical_cols = [c for c in df.select_dtypes(exclude="number").columns if c != TARGET]

print("Numeric features    :", numeric_cols)
print("Categorical features:", categorical_cols)
print("Target              :", TARGET)

Numeric features    : ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
Categorical features: ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']
Target              : income


In [5]:
# Summary statistics for numeric features
df.describe().T

,count,mean,std,min,25%,50%,75%,max
age,48842.0,38.643585,13.710510,17.0,28.0,37.0,48.0,90.0
fnlwgt,48842.0,189664.134597,105604.025423,12285.0,117550.5,178144.5,237642.0,1490400.0
education-num,48842.0,10.078089,2.570973,1.0,9.0,10.0,12.0,16.0
capital-gain,48842.0,1079.067626,7452.019058,0.0,0.0,0.0,0.0,99999.0
capital-loss,48842.0,87.502314,403.004552,0.0,0.0,0.0,0.0,4356.0
hours-per-week,48842.0,40.422382,12.391444,1.0,40.0,40.0,45.0,99.0


### 3.2 Data quality checks

In [6]:
# 1) Missing values: the UCI files encode "unknown" as '?', so pandas does not see them as NaN
question_marks = (df == "?").sum()
print("Cells containing '?':")
print(question_marks[question_marks > 0])
print(f"\nRows with at least one '?': {(df == '?').any(axis=1).sum():,} "
      f"({(df == '?').any(axis=1).mean():.1%} of rows)")
print(f"True NaN values: {df.isna().sum().sum()}")

Cells containing '?':
workclass         2799
occupation        2809
native-country     857
dtype: int64

Rows with at least one '?': 3,620 (7.4% of rows)
True NaN values: 0


In [7]:
# 2) Inconsistent target labels: the test file adds a trailing period ('>50K.')
df[TARGET].value_counts()

income
<=50K     24720
<=50K.    12435
>50K       7841
>50K.      3846
Name: count, dtype: int64

In [8]:
# 3) Exact duplicate rows
print(f"Duplicate rows: {df.duplicated().sum():,}")

Duplicate rows: 29


In [9]:
# 4) Skewed numeric features: capital gain / loss are zero for most people
for col in ["capital-gain", "capital-loss"]:
    print(f"{col}: {(df[col] == 0).mean():.1%} zeros, max = {df[col].max():,}")

capital-gain: 91.7% zeros, max = 99,999
capital-loss: 95.3% zeros, max = 4,356


In [10]:
# 5) Redundant columns: education (text) and education-num (ordinal code) carry the same information
codes_per_level = df.groupby("education")["education-num"].nunique()
print(f"Each education level maps to exactly one education-num code: {(codes_per_level == 1).all()}")
df.groupby("education")["education-num"].first().sort_values().to_frame().T

Each education level maps to exactly one education-num code: True


education,Preschool,1st-4th,5th-6th,7th-8th,9th,10th,11th,12th,HS-grad,Some-college,Assoc-voc,Assoc-acdm,Bachelors,Masters,Prof-school,Doctorate
education-num,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16


### 3.3 Data quality notes

- **Hidden missing values.** Missing entries are stored as `'?'` in `workclass`, `occupation` and `native-country` (7.4% of rows affected), not as `NaN`. They must be converted before imputation.
- **Inconsistent labels.** The target has four spellings (`<=50K`, `<=50K.`, `>50K`, `>50K.`) because the original test file appends a period. These will be normalized to two classes.
- **Duplicates.** 29 exact duplicate rows exist (<0.1%). With no person identifier, they could be genuinely different people with identical attributes, so this is noted rather than assumed to be an error.
- **Skewed features.** `capital-gain` and `capital-loss` are zero for the vast majority of rows, with a long right tail; the maximum of 99,999 looks like a recording cap rather than a real value.
- **Redundancy.** `education` and `education-num` map one-to-one, so only one should be kept. `fnlwgt` is a census sampling weight rather than a personal attribute.

*Next: exploratory analysis and preprocessing (Section 4).*